# **"Ask bard" agent**

## **Boilerplate**

In [79]:
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ["OPENAI_API_KEY"]:
    print("OpenAI api key set")
else:
    raise ValueError("OpenAI api key is not set")

OpenAI api key set


In [80]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4")

## **Breif Description**

---

In a song screen, you will be able to highlight a part of the song that you want to know the theory behind and click the "bardify" button. This will then add commentary to the tabs and give smart vidual aids for you to help understand whats going on:

Scales: if part of the section includes notes of a certain scale, the whole scale shape will light up on the fretboard in a different color so you can see the notes of the song outline it.

Chord progressions: if the part follows a certain chord progression, you will see a roman numeral chart light up corresponding to which chord is being played at a given moment.

these are two preliminary ideas, there will be many more similar features added but these two feel like a good starting place.

## **Create Graph Schema**

In [81]:
from pydantic import BaseModel, Field
from typing import List, Annotated
import operator

class graph_schema(BaseModel):
    measures: List = Field(description="a list of the measures to be analyzed")
    underlying_scale: List = Field(description="the underlying scale being outlined during the selected measures. Stored in the same format as the measures themselves")

## **Create llm Schema**

In [82]:
class llm_schema(BaseModel):
    underlying_scale: str = Field(description="the underlying scale being outlined during the selected measures. Stored in the same format as the measures themselves")

llm_with_schema = llm.with_structured_output(llm_schema)

llm_with_schema.invoke("give one measure of the pentatonic scale as a list of beats nested inside a measure dict")

llm_schema(underlying_scale="{'measure': [['C4'], ['D4'], ['E4'], ['G4'], ['A4']]}")

## **Create Node Functions**

In [83]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage, HumanMessage
import json


def determine_scale(state: graph_schema) -> graph_schema:

    measures = ", ".join(json.dumps(measure) for measure in state.measures)

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a guitar music theory expert who looks at the notes of a song and determines the scales being played"),
            ("human", "Determine the scale being played in this song:\n\n{measures}\n\nThen put all the notes of the underlying scale shape into your response in the same format as the input.")
        ]
    )

    chain = prompt | llm_with_schema

    response = chain.invoke({"measures": measures})

    state.underlying_scale = response.underlying_scale

    return state



## **Get some notes and key for testing**

In [84]:


test_tempo = 100
test_measures = [
    {
        "index": 0,
        "beats": [
            {"time": 0.0, "duration": 0.6, "notes": [{"string": 2, "fret": 1, "midi": 60}]},   # C4
            {"time": 0.6, "duration": 0.6, "notes": [{"string": 2, "fret": 3, "midi": 62}]},   # D4
            {"time": 1.2, "duration": 0.6, "notes": [{"string": 1, "fret": 0, "midi": 64}]},   # E4
            {"time": 1.8, "duration": 0.6, "notes": [{"string": 1, "fret": 1, "midi": 65}]},   # F4
        ],
    },
    {
        "index": 1,
        "beats": [
            {"time": 2.4, "duration": 0.6, "notes": [{"string": 1, "fret": 3, "midi": 67}]},   # G4
            {"time": 3.0, "duration": 0.6, "notes": [{"string": 1, "fret": 5, "midi": 69}]},   # A4
            {"time": 3.6, "duration": 0.6, "notes": [{"string": 1, "fret": 7, "midi": 71}]},   # B4
            {"time": 4.2, "duration": 0.6, "notes": [{"string": 1, "fret": 8, "midi": 72}]},   # C5
        ],
    },
]

with open("e_minor_pentatonic.txt", "r", encoding="utf-8") as file:
    z_minor_pentatonic=json.loads(file.read())



## **Create Graph**

In [85]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(graph_schema)

graph.add_node("determine_scale", determine_scale)

graph.add_edge(START, "determine_scale")
graph.add_edge("determine_scale", END)

test_graph = graph.compile()


## **Invoke Graph**

In [86]:
test_graph.invoke({"measures": z_minor_pentatonic, "underlying_scale": []})

{'measures': [{'beats': [{'time': 0.0,
     'notes': [{'fret': 0, 'midi': 40, 'string': 6}],
     'duration': 0.3},
    {'time': 0.3,
     'notes': [{'fret': 3, 'midi': 43, 'string': 6}],
     'duration': 0.3},
    {'time': 0.6,
     'notes': [{'fret': 0, 'midi': 45, 'string': 5}],
     'duration': 0.3},
    {'time': 0.9,
     'notes': [{'fret': 2, 'midi': 47, 'string': 5}],
     'duration': 0.3},
    {'time': 1.2,
     'notes': [{'fret': 0, 'midi': 50, 'string': 4}],
     'duration': 0.3},
    {'time': 1.5,
     'notes': [{'fret': 2, 'midi': 52, 'string': 4}],
     'duration': 0.3},
    {'time': 1.8,
     'notes': [{'fret': 0, 'midi': 55, 'string': 3}],
     'duration': 0.3},
    {'time': 2.1,
     'notes': [{'fret': 2, 'midi': 57, 'string': 3}],
     'duration': 0.3}],
   'index': 0},
  {'beats': [{'time': 2.4,
     'notes': [{'fret': 0, 'midi': 59, 'string': 2}],
     'duration': 0.3},
    {'time': 2.7,
     'notes': [{'fret': 3, 'midi': 62, 'string': 2}],
     'duration': 0.3},
   